                # CSE543 Information Assurance and Security - Fuzz Them All

                Created with Grok Build

                The submission was a mutational fuzzer written in C so the loop would be fast. It read a seed into a buffer. On each iteration it replaced a byte with a random value from 1 to 255 about 13 percent of the time. Every 500 iterations it appended ten more random bytes, then wrote the buffer out. A separate runner fed those buffers to the course programs and watched for a crash.

                This notebook is a new sketch of that shape. It does not include the course programs, the seeds that crashed them, or the runner that launched them.
                


## One mutation pass


In [1]:
import random


def mutate(buf, rng, percent=13):
    out = bytearray(buf)
    changed = []
    for index in range(len(out)):
        if rng.randrange(100) < percent:
            out[index] = rng.randrange(1, 256)
            changed.append(index)
    return bytes(out), changed


rng = random.Random(7)
seed = b"hello"
mutated, changed = mutate(seed, rng)
print("seed length", len(seed))
print("changed indexes", changed)
print("mutated length", len(mutated))


seed length 5
changed indexes [4]
mutated length 5


                ## How often a byte changes

                Over a thousand coin-flips at 13 percent, the share of flips that win should sit near 0.13. That is the mutation rate from the C fuzzer, measured here without touching a target program.
                


In [2]:
rng = random.Random(7)
wins = sum(rng.randrange(100) < 13 for _ in range(1000))
print(wins, "of 1000")
print(f"{wins / 1000:.3f}")


141 of 1000
0.141


                ## Growing the buffer

                The C fuzzer appended ten bytes whenever the iteration count was a multiple of 500. The loop below only tracks length. The appended bytes are a placeholder, not a crash input.
                


In [3]:
buf = b"seed"
for iteration in range(1, 1001):
    if iteration % 500 == 0:
        buf = buf + bytes([1]) * 10
        print(iteration, len(buf))


500 14
1000 24
